<a href="https://colab.research.google.com/github/RvXp/Topicos-Especiais-em-IA-LLM/blob/main/Finetune_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4 — Fine-tuning de um Transformer Decoder-only

Nesta prática faremos um **fine-tuning supervisionado (SFT)** em um **decoder-only causal LM** usando o menor modelo *útil* e popular para demos no Hugging Face (https://huggingface.co/distilbert/distilgpt2):

- **Modelo base:** `distilbert/distilgpt2`

Passos:
1. Carregar/definir documentos (toy ou pasta local)
2. Tokenização + *grouping* em blocos (CLM)
3. Fine-tuning com `Trainer`
4. Inferência

In [5]:
# (Opcional) Instalar dependências no ambiente local/Colab
!pip -q install transformers datasets accelerate

In [6]:
import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch: 2.10.0+cpu
cuda available: False


## 1) Dataset — Documentos (toy) ou pasta local

### Opção A (didática): documentos toy
### Opção B (real): ler `.txt` de uma pasta local

> Para aula, a Opção A roda rápido e mostra o pipeline completo.


In [7]:
# Opção A: documentos toy
docs = [
    "O homem é a medida de todas as coisas, das coisas que são, enquanto são, das coisas que não são, enquanto não são.",
    "Só sei que nada sei.",
    "A vida não examinada não vale a pena ser vivida.",
    "Ninguém entra no mesmo rio duas vezes, pois tudo flui e nada permanece.",
    "O ser é e o não-ser não é.",
    "A virtude é o único bem; o vício, o único mal.",
    "Onde há amor e sabedoria, não há medo nem ignorância.",
    "A felicidade depende de nós mesmos.",
    "O homem é um animal político.",
    "A coragem é a primeira das qualidades humanas, porque garante todas as outras.",
    "Não espere que o mundo seja como você deseja, mas deseje que ele seja como é.",
    "A morte não é nada para nós, pois, quando existimos, não há morte; e quando há morte, não existimos.",
    "O desejo é a medida de todas as nossas ações.",
    "A mente é como uma tábula rasa, onde a experiência escreve.",
    "Penso, logo existo.",
    "O coração tem razões que a própria razão desconhece.",
    "O homem nasceu livre, e em toda parte se encontra acorrentado.",
    "Aja apenas segundo a máxima que você gostaria que se tornasse uma lei universal.",
    "Deus está morto e nós o matamos.",
    "A existência precede a essência.",
    "O inferno são os outros.",
    "O homem está condenado a ser livre.",
    "A angústia é a vertigem da liberdade.",
    "O absurdo nasce do confronto entre o apelo humano e o silêncio irracional do mundo.",
    "É preciso imaginar Sísifo feliz.",
    "O eterno retorno é a mais pesada das cargas.",
    "A história de toda a sociedade até agora é a história da luta de classes.",
    "Os filósofos apenas interpretaram o mundo de diferentes maneiras; a questão, porém, é transformá-lo.",
    "A linguagem é a casa do ser.",
    "Os limites da minha linguagem significam os limites do meu mundo.",
    "Sobre o que não se pode falar, deve-se calar.",
    "A verdade é a adequação do intelecto à coisa.",
    "O conhecimento é poder.",
    "A religião é o ópio do povo.",
    "O homem é o lobo do homem.",
    "A justiça sem força é impotente, a força sem justiça é tirânica.",
    "Duvidar de tudo é o primeiro passo para a verdade.",
    "Não há fatos, apenas interpretações.",
    "Torna-te quem tu és.",
    "O que não me mata me fortalece.",
    "A vontade de potência é a essência da vida.",
    "O super-homem é o sentido da terra.",
    "A moralidade é o instinto de rebanho no indivíduo.",
    "O universo não é apenas mais estranho do que supomos, mas mais estranho do que podemos supor.",
    "Tudo o que é racional é real, e tudo o que é real é racional.",
    "A coruja de Minerva levanta voo apenas ao entardecer.",
    "A liberdade é a consciência da necessidade.",
    "A alienação é a separação do homem de sua própria essência.",
    "A cultura é um sistema de símbolos.",
    "O poder não é algo que se possui, mas algo que se exerce.",
    "A loucura é a ausência de obra.",
    "O panóptico é a utopia de uma sociedade perfeitamente governada.",
    "O gênero é uma performance, não uma essência.",
    "A banalidade do mal é o perigo da burocracia sem pensamento.",
    "A condição humana é a pluralidade.",
    "O véu da ignorância garante a justiça.",
    "A liberdade negativa é a ausência de impedimentos.",
    "A liberdade positiva é a capacidade de autodeterminação.",
    "O fim da história é o triunfo da democracia liberal.",
    "O choque de civilizações dominará a política global.",
    "A sociedade líquida não mantém formas por muito tempo.",
    "O amor líquido é a fragilidade dos laços humanos.",
    "O simulacro é a verdade que esconde que não há verdade.",
    "A hiper-realidade é mais real que o real.",
    "A desconstrução é a justiça.",
    "Não há fora-do-texto.",
    "O rosto do outro me interpela e me faz responsável.",
    "A ética é a filosofia primeira.",
    "O rizoma não tem começo nem fim, mas sempre um meio.",
    "O corpo sem órgãos é o campo de imanência do desejo.",
    "A biopolítica é a gestão da vida biológica pela política.",
    "O estado de exceção tornou-se a regra.",
    "A vida nua é a vida exposta à morte sem proteção jurídica.",
    "O orientalismo é um estilo ocidental de dominação.",
    "A dialética do esclarecimento mostra como a razão se torna mito.",
    "A indústria cultural engana as massas.",
    "O homem unidimensional perdeu a capacidade de crítica.",
    "A esfera pública é o espaço do debate racional.",
    "A ação comunicativa busca o entendimento, não o sucesso estratégico.",
    "A ciência normal é a resolução de quebra-cabeças dentro de um paradigma.",
    "As revoluções científicas são mudanças de paradigma.",
    "A falseabilidade é o critério de demarcação da ciência.",
    "Tudo vale na epistemologia anarquista.",
    "O mito do dado é a ilusão de que a experiência é imediata.",
    "A mente não está na cabeça, mas na interação com o mundo.",
    "Qualia são as qualidades subjetivas da experiência consciente.",
    "O problema difícil da consciência é explicar como a matéria gera o sentir.",
    "O zumbi filosófico é fisicamente idêntico, mas sem consciência.",
    "O quarto chinês mostra que sintaxe não é semântica.",
    "Ser é ser percebido.",
    "As mônadas são as substâncias simples que compõem o universo.",
    "Vivemos no melhor dos mundos possíveis.",
    "O espaço e o tempo são formas a priori da sensibilidade.",
    "A coisa em si é incognoscível.",
    "A beleza é a promessa de felicidade.",
    "O sublime é o prazer que nasce do terror.",
    "A ironia é a atitude de quem sabe que não sabe.",
    "A dialética é o movimento do espírito.",
    "O senhor e o escravo lutam pelo reconhecimento.",
    "A ideologia é a falsa consciência da realidade.",
    "O fetiche da mercadoria transforma relações sociais em relações entre coisas.",
    "O capital é trabalho morto que, como um vampiro, vive sugando trabalho vivo.",
    "A utopia é o horizonte que nos faz caminhar.",
    "O pessimista é um otimista bem informado.",
    "A vida oscila como um pêndulo entre a dor e o tédio.",
    "A compaixão é a base da moralidade.",
    "O mundo é minha representação.",
    "O eterno silêncio desses espaços infinitos me apavora.",
    "O homem é um caniço, o mais fraco da natureza, mas é um caniço pensante.",
    "A aposta de Pascal sugere que crer é a opção mais racional.",
    "O contrato social legitima a autoridade política.",
    "O estado de natureza é a guerra de todos contra todos.",
    "A propriedade é o roubo.",
    "A anarquia é a ordem.",
    "O utilitarismo busca a maior felicidade para o maior número.",
    "É melhor ser um humano insatisfeito do que um porco satisfeito.",
    "A liberdade de um termina onde começa a do outro.",
    "A mão invisível regula o mercado.",
    "A mais-valia é a fonte do lucro.",
    "A revolução é a locomotiva da história.",
    "O ser-para-a-morte é a autenticidade do Dasein.",
    "A técnica é a forma moderna de desvelamento do ser.",
    "A hermenêutica é a arte da interpretação.",
    "O círculo hermenêutico implica que o todo depende das partes e vice-versa.",
    "A fusão de horizontes é o objetivo da compreensão histórica.",
    "A tradição é a autoridade que nos antecede.",
    "O preconceito não é necessariamente negativo, mas uma pré-compreensão.",
    "A desconstrução revela as hierarquias ocultas no texto.",
    "A diferença é o que difere e adia o sentido.",
    "A arqui-escritura precede a fala.",
    "O falogocentrismo domina a filosofia ocidental.",
    "O corpo é o lugar da inscrição do poder.",
    "A microfísica do poder atua nos corpos e nas almas.",
    "A confissão é a técnica de produção da verdade sobre o sujeito.",
    "O cuidado de si é a ética da liberdade.",
    "A heterotopia é o espaço outro, real e utópico ao mesmo tempo.",
    "O rizoma conecta qualquer ponto a qualquer outro ponto.",
    "A desterritorialização é o movimento de fuga.",
    "A máquina de guerra é exterior ao Estado.",
    "O plano de imanência é o absoluto da filosofia.",
    "O conceito é o que a filosofia cria.",
    "A multidão é o sujeito da pós-modernidade.",
    "O império é a nova ordem global sem fronteiras.",
    "A vida nua é a política do campo de concentração.",
    "O homo sacer é aquele que pode ser morto, mas não sacrificado.",
    "A comunidade que vem não tem identidade.",
    "A espectrologia estuda os fantasmas da história.",
    "O realismo especulativo rejeita o correlacionismo.",
    "O objeto orientado para si mesmo retira-se da relação.",
    "A hiperstição é a ficção que se torna realidade.",
    "O Tao que pode ser nomeado não é o Tao eterno.",
    "O sábio age pelo não-agir.",
    "A suavidade vence a dureza.",
    "O vazio é a utilidade do vaso.",
    "Tudo é impermanente, tudo é sofrimento, tudo é não-eu.",
    "O apego é a raiz do sofrimento.",
    "O nirvana é a extinção da chama do desejo.",
    "A forma é vazio, o vazio é forma.",
    "O zen é a mente de principiante.",
    "Se encontrar o Buda na estrada, mate-o.",
    "A compaixão é a essência do Bodhisattva.",
    "O karma é a lei de causa e efeito moral.",
    "O eu é uma ilusão criada pelos agregados.",
    "A meditação é o caminho para a visão clara.",
    "A não-dualidade transcende sujeito e objeto.",
    "Brahman é a realidade última, Atman é o eu verdadeiro.",
    "Tu és isso (Tat Tvam Asi).",
    "O mundo é Maya, ilusão cósmica.",
    "O dharma é o dever cósmico e social.",
    "A ioga é a cessação das flutuações da mente.",
    "A piedade filial é a raiz da humanidade.",
    "O nobre não é um utensílio.",
    "Governe através da virtude, e será como a estrela polar.",
    "A retificação dos nomes é essencial para a ordem."
]
ds = Dataset.from_dict({"text": docs}).train_test_split(test_size=0.1, seed=42)
ds

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 156
    })
    test: Dataset({
        features: ['text'],
        num_rows: 18
    })
})

In [8]:
# Opção B (descomente se quiser ler arquivos .txt de uma pasta):
# from glob import glob
# paths = glob("./meus_docs/*.txt")
# docs = [open(p, "r", encoding="utf-8").read() for p in paths]
# ds = Dataset.from_dict({"text": docs}).train_test_split(test_size=0.1, seed=42)
# ds

## 2) Tokenizer + Modelo base

Usaremos o tokenizer do modelo base e definiremos `pad_token`, pois GPT-2 não tem por padrão.


In [9]:
model_id = "distilbert/distilgpt2"

tok = AutoTokenizer.from_pretrained(model_id)

# GPT-2 não define pad_token por padrão; necessário para batches com padding
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print("Vocab size:", len(tok))
print("Model loaded:", model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Vocab size: 50257
Model loaded: distilbert/distilgpt2


## 3) Tokenização + agrupamento em blocos (Causal LM)

Em *Causal Language Modeling*, a label é o próprio `input_ids` deslocado internamente pelo modelo.
Uma forma padrão é concatenar tudo e quebrar em blocos fixos (`block_size`).


In [10]:
block_size = 64  # pequeno para rodar rápido

def tokenize_fn(batch):
    return tok(batch["text"], truncation=True)

tok_ds = ds.map(tokenize_fn, batched=False, remove_columns=["text"])

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated["input_ids"])
    print('total_len = ', total_length)
    total_length = (total_length // block_size) * block_size
    print('total_len = ', total_length)
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_ds = tok_ds.map(group_texts, batched=True)
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

Map:   0%|          | 0/156 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Map:   0%|          | 0/156 [00:00<?, ? examples/s]

total_len =  2995
total_len =  2944


Map:   0%|          | 0/18 [00:00<?, ? examples/s]

total_len =  287
total_len =  256


## 4) Fine-tuning (SFT) com `Trainer`

> Com o seguinte código, abstraímos a parte do treino e executar 10 épocas para o modelo aprender o conteúdo do novo corpus.


In [11]:
args = TrainingArguments(
    output_dir="./ft_tiny_gpt2",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=10,
    learning_rate=5e-4,
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=lm_ds["train"],
    eval_dataset=lm_ds["test"],
    data_collator=collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,4.531535,4.264064
2,3.066434,4.293593
3,2.660402,5.031184
4,2.189759,5.181833
5,1.658499,5.673679
6,1.179431,6.433116
7,0.745052,7.058531
8,0.495425,7.735731
9,0.339743,7.946068
10,0.183128,8.009958


TrainOutput(global_step=230, training_loss=1.7203322234361067, metrics={'train_runtime': 483.5917, 'train_samples_per_second': 0.951, 'train_steps_per_second': 0.476, 'total_flos': 7512281579520.0, 'train_loss': 1.7203322234361067, 'epoch': 10.0})

## 5) Inferência depos do finetune



In [12]:
prompt=tok("UFU ", return_tensors = "pt", truncation = True).input_ids[0,]
inputs = prompt.unsqueeze(0).to(device)
with torch.no_grad():
    out_ids = model.generate(
        inputs,
        max_new_tokens=10,
        do_sample=True,
        pad_token_id=tok.eos_token_id,
        temperature= 0.7,
        top_k = 50,
        top_p = 1.0
    )
print(f"Inference: {tok.decode(out_ids[0])}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Inference: UFU ética do das partes e vice-


In [13]:
print("\n--- Inferência em Documentos de Treino ---")
train_docs = ds['train']['text']

for i, doc_text in enumerate(train_docs):
    initial_tokens = tok(doc_text, return_tensors="pt", truncation=True).input_ids[0, :10]
    inputs = initial_tokens.unsqueeze(0).to(device) # unsqueeze to add batch dimension

    with torch.no_grad():
        out_ids = model.generate(
            inputs,
            max_new_tokens=10,
            do_sample=True,
            pad_token_id=tok.eos_token_id,
            temperature= 0.7,
            top_k = 50,
            top_p = 1.0
        )

    # Decode the sequence
    prompt_text = tok.decode(initial_tokens, skip_special_tokens=True)
    generated_text = tok.decode(out_ids[0], skip_special_tokens=True)
    print(f"doc{i}: {prompt_text}*{generated_text[len(prompt_text):]}*")



--- Inferência em Documentos de Treino ---
doc0: O nobre não é um utensí*lio.É melhor ser um humano ins*
doc1: É melhor ser um humano insatisfe*ito.A história é a socied*
doc2: Tu és isso (Tat Tv*am Asi).A anarquia é*
doc3: A coragem é a primeira das* parte se torna mito.Não*
doc4: Vivemos no melhor dos mundos poss*ui.A ação é a té*
doc5: O pessimista é um otimista bem*; otimista bem; otimista*
doc6: A heterotopia é o espaço* a própria razão descon*
doc7: A piedade filial é a raiz* da humanidade.A poder até*
doc8: Tudo é impermanente, tudo é* sofrimento, tudo é não*
doc9: O sábio age pelo não* é aquele que pode ser nome*
doc10: Ninguém entra no mesmo r*azões que a própria*
doc11: Qualia são as qualidades subjet*ivas da experiência consciente*
doc12: O rosto do outro me interp*ela e me mais real que o real que*
doc13: A indústria cultural engana as* substâncias simples que comp�*
doc14: O homem é um animal político*.A forma é a essência*
doc15: Torna-te quem tu és*.A ética é a ética*
d

# Exercício prático

Escolha um prompt que não estava nos dados de treino, mas que seja do mesmo domínio. Faça comparações. Documente a saída do modelo base (sem treino) e do modelo ajustado. Verifique se o modelo "aprendeu" termos técnicos ou o vocabulário específico que você forneceu nos arquivos .txt.

In [15]:
# O objeto 'model' atual já é o seu modelo treinado (Fine-tuned)
model_ajustado = model

# Carregamos uma versão limpa do modelo para comparação (Base)
model_base = AutoModelForCausalLM.from_pretrained(model_id).to(device)

# Lista de frases de teste (domínio filosófico, mas fora do treino)
frases_teste = [
    "A prudência é ",
    "A liberdade é ",
    "O inferno é ",
    "Se Deus não existe, "
]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilbert/distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def comparar_modelos(prompt_text):
    inputs = tok(prompt_text, return_tensors="pt").to(device)

    # Configurações de geração padrão do seu exercício
    gen_args = {
        "max_new_tokens": 12,
        "do_sample": True,
        "temperature": 0.7,
        "top_k": 50,
        "pad_token_id": tok.eos_token_id
    }

    with torch.no_grad():
        # Geração com o modelo original
        out_base = model_base.generate(**inputs, **gen_args)
        # Geração com o seu modelo treinado
        out_ft = model_ajustado.generate(**inputs, **gen_args)

    print(f"\nPROMPT: {prompt_text}")
    print(f"RESPOSTA BASE:    {tok.decode(out_base[0], skip_special_tokens=True)}")
    print(f"RESPOSTA AJUSTADA: {tok.decode(out_ft[0], skip_special_tokens=True)}")
    print("-" * 30)

# Executa para as novas frases
for frase in frases_teste:
    comparar_modelos(frase)


PROMPT: A prudência é 
RESPOSTA BASE:    A prudência é état. la fait d’état d
RESPOSTA AJUSTADA: A prudência é és o instinto de rebanho no indiví
------------------------------

PROMPT: A liberdade é 
RESPOSTA BASE:    A liberdade é été.”
The English word “l
RESPOSTA AJUSTADA: A liberdade é és.A ética é a ética da
------------------------------

PROMPT: O inferno é 
RESPOSTA BASE:    O inferno é été délèbre à héctoré de
RESPOSTA AJUSTADA: O inferno é ésofia caminhar.A história é
------------------------------

PROMPT: Se Deus não existe, 
RESPOSTA BASE:    Se Deus não existe, été que été de das de auc
RESPOSTA AJUSTADA: Se Deus não existe, és.A utopia é o instinto de reban
------------------------------
